# Lesson 07: The Gemma SLM Family — MatFormer, Per-Layer Embeddings (PLE) & Mobile Multimodal AI

In this companion notebook, you will:
1. Model **Per-Layer Embeddings (PLE)** SRAM offloading savings for mobile-first `Gemma 3n` E2B and E4B architectures.
2. Inspect **5:1 local-to-global sliding-window attention** KV-cache savings for 128K context windows.
3. Run an open-source mobile-scale transformer forward/generation check with `transformers` on Colab.

In [ ]:
!pip install -q -U transformers accelerate torch

## 1. Per-Layer Embeddings (PLE) & Sliding-Window KV-Cache Calculator
Gemma 3 reduces KV-cache explosion at long contexts using a 5:1 ratio of local sliding-window layers (1,024-token window) to global attention layers, while Gemma 3n uses PLE to keep NPU/GPU SRAM under 2-3 GB.

In [ ]:
def kv_cache_ratio_5_to_1(num_layers: int, ctx_len: int, window_size: int = 1024) -> float:
    global_layers = num_layers // 6
    local_layers = num_layers - global_layers
    baseline_tokens = num_layers * ctx_len
    hybrid_tokens = global_layers * ctx_len + local_layers * min(ctx_len, window_size)
    return hybrid_tokens / baseline_tokens

for ctx in [4096, 32768, 131072]:
    ratio = kv_cache_ratio_5_to_1(30, ctx)
    print(f"Context {ctx:6d} tokens -> Hybrid 5:1 KV cache is {ratio*100:.1f}% of full global attention")

## 2. Run Open-Access Gemma-Scale Tokenizer & Generation Check
Note: Gated `google/gemma-3-*` checkpoints require accepting the license on Hugging Face and setting `HF_TOKEN`. Below we verify with an open-access compact model and fall back cleanly if `HF_TOKEN` is not set.

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token = os.environ.get("HF_TOKEN")
model_id = "google/gemma-3-1b-it" if hf_token else "HuggingFaceTB/SmolLM2-360M-Instruct"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(model_id, token=hf_token)
prompt = "Explain Per-Layer Embeddings for mobile AI in two sentences."
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))